# Jacobian Lens / J-Space Colab Walkthrough

이 노트북은 Anthropic의 *Verbalizable Representations Form a Global Workspace in Language Models* 논문과 공개 `jacobian-lens` 참조 구현을 작은 open-weight 모델에서 따라가는 튜토리얼입니다.

기본 경로는 **이미 fitting된 Neuronpedia lens를 불러와 적용**하는 것입니다. 전체 Neuronpedia 웹 서비스나 데이터베이스는 설치하지 않습니다.

## Goal

1. 같은 residual activation을 J-lens와 logit lens로 읽습니다.
2. layer × token position에 나타나는 top concept를 비교합니다.
3. 관심 token의 vocabulary rank가 layer에 따라 어떻게 변하는지 봅니다.
4. 논문의 pseudoinverse coordinate swap을 작은 모델에 적용해 다음-token 분포 변화를 측정합니다.

> 권장 런타임: Colab GPU(T4 이상). `런타임 → 런타임 유형 변경 → GPU`를 선택하세요.
> 작은 open-weight 모델은 논문의 Claude 결과를 재현하는 대상이 아니라, 공개 방법의 작동 원리를 검증하는 대상입니다.

## Setup

Colab 세션마다 한 번 실행합니다. 설치 후 import 오류가 나면 `런타임 다시 시작` 후 다음 셀부터 실행하세요.

In [ ]:
%pip -q install "git+https://github.com/anthropics/jacobian-lens.git" "transformers>=5.5" "huggingface_hub>=0.30" pandas seaborn matplotlib

In [ ]:
import gc
import math
import random
from contextlib import contextmanager

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import transformers
import jlens
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    major, _ = torch.cuda.get_device_capability()
    MODEL_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.float32

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("device:", DEVICE, "dtype:", MODEL_DTYPE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("경고: GPU가 아니므로 실행이 느릴 수 있습니다.")

### Parameters

기본값은 Colab T4에서도 비교적 가벼운 Qwen 0.8B입니다. 더 빠른 smoke test가 필요하면 아래의 `FAST_SMOKE_TEST = True`로 바꾸세요.

In [ ]:
FAST_SMOKE_TEST = False

if FAST_SMOKE_TEST:
    MODEL_ID = "EleutherAI/pythia-70m-deduped"
    LENS_FILENAME = (
        "pythia-70m-deduped/jlens/Salesforce-wikitext/"
        "pythia-70m-deduped_jacobian_lens.pt"
    )
else:
    MODEL_ID = "Qwen/Qwen3.5-0.8B"
    LENS_FILENAME = (
        "qwen3.5-0.8b/jlens/Salesforce-wikitext/"
        "Qwen3.5-0.8B_jacobian_lens.pt"
    )

LENS_REPO = "neuronpedia/jacobian-lens"
PROMPT = "The number of legs on the animal that spins webs is"
TOP_K = 8
MAX_DISPLAY_LAYERS = 12

print("model:", MODEL_ID)
print("lens:", f"{LENS_REPO}/{LENS_FILENAME}")
print("prompt:", PROMPT)

### 1. Load the model and pre-fitted lens

Lens 파일에는 layer별 평균 Jacobian `J_l`이 fp16으로 저장되어 있습니다. 모델과 lens의 `d_model`이 반드시 같아야 합니다.

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True
)
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
hf_model.eval()

lens_model = jlens.from_hf(hf_model, tokenizer, compile=False)
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILENAME
)

assert lens.d_model == lens_model.d_model, (lens.d_model, lens_model.d_model)
print(lens_model)
print(lens)
print("fitted prompts:", lens.n_prompts)
print("source layers:", lens.source_layers)

## Steps

### 2. Select layers and inspect tokenization

전체 layer를 모두 unembed하면 vocabulary 크기만큼의 logits가 생깁니다. 메모리를 제한하기 위해 layer를 균등하게 subsample합니다.

In [ ]:
encoded = tokenizer(PROMPT, return_tensors="pt", add_special_tokens=True)
input_ids_cpu = encoded.input_ids[0]
position_labels = [
    f"{i}:{tokenizer.decode([int(token_id)])!r}"
    for i, token_id in enumerate(input_ids_cpu)
]

layer_stride = max(1, math.ceil(len(lens.source_layers) / MAX_DISPLAY_LAYERS))
display_layers = lens.source_layers[::layer_stride]
if lens.source_layers[-1] not in display_layers:
    display_layers.append(lens.source_layers[-1])

print("token positions:")
print(" | ".join(position_labels))
print("display layers:", display_layers)

### 3. Run J-lens and logit lens

두 호출은 동일한 모델을 각각 한 번 forward합니다. J-lens는 residual에 `J_l`을 적용한 뒤 unembed하고, logit lens는 residual을 바로 unembed합니다.

In [ ]:
with torch.inference_mode():
    j_logits, final_logits, input_ids = lens.apply(
        lens_model,
        PROMPT,
        layers=display_layers,
        positions=None,
        use_jacobian=True,
    )
    logit_logits, _, _ = lens.apply(
        lens_model,
        PROMPT,
        layers=display_layers,
        positions=None,
        use_jacobian=False,
    )

seq_len = int(input_ids.shape[1])
assert all(tensor.shape[0] == seq_len for tensor in j_logits.values())
assert all(tensor.shape == j_logits[layer].shape for layer, tensor in logit_logits.items())
print("sequence length:", seq_len)
print("vocabulary size:", next(iter(j_logits.values())).shape[-1])
print("readout checks: OK")

### 4. Compare top concepts at the final prompt position

J-lens가 중간 layer에서 `spider` 같은 출력되지 않은 intermediate를 올리는지 확인합니다. 작은 모델에서는 나타나지 않을 수도 있으며, 그 자체가 유효한 결과입니다.

In [ ]:
def decode_topk(logits_row, k=TOP_K):
    values, indices = logits_row.float().topk(k)
    return [
        tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)
        for token_id in indices
    ]

selected_position = seq_len - 1
comparison_rows = []
for layer in display_layers:
    comparison_rows.append(
        {
            "layer": layer,
            "J-lens top tokens": decode_topk(j_logits[layer][selected_position]),
            "logit-lens top tokens": decode_topk(logit_logits[layer][selected_position]),
        }
    )
display(pd.DataFrame(comparison_rows))

### 5. Layer × position top-1 table

각 cell은 해당 token position과 layer에서 J-lens가 가장 높게 읽은 vocabulary token입니다.

In [ ]:
top1_table = pd.DataFrame(index=position_labels)
for layer in display_layers:
    top_ids = j_logits[layer].argmax(dim=-1)
    top1_table[f"L{layer}"] = [
        tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)
        for token_id in top_ids
    ]
display(top1_table.T)

### 6. Track selected token ranks

Token 문자열은 tokenizer에 따라 하나 또는 여러 token으로 분리될 수 있습니다. 아래 코드는 먼저 단일-token 후보만 선택한 뒤, J-lens와 logit lens의 `log10(rank)`를 비교합니다. 낮을수록 강하게 읽힌 것입니다.

In [ ]:
PINNED_SURFACES = [" spider", " ant", " 8", " 6", " animal"]

def resolve_single_token(surface):
    token_ids = tokenizer.encode(surface, add_special_tokens=False)
    if len(token_ids) != 1:
        print(f"skip {surface!r}: tokenized as {token_ids}")
        return None
    return int(token_ids[0])

pinned = {surface: resolve_single_token(surface) for surface in PINNED_SURFACES}
pinned = {surface: token_id for surface, token_id in pinned.items() if token_id is not None}
assert pinned, "단일-token PINNED_SURFACES가 하나도 없습니다. 후보를 변경하세요."
print(pinned)

def vocabulary_rank(logits_row, token_id):
    target = logits_row[token_id]
    return int((logits_row > target).sum().item()) + 1

def rank_frame(layer_logits):
    rows = []
    for layer in display_layers:
        row = {"layer": layer}
        logits_row = layer_logits[layer][selected_position]
        for surface, token_id in pinned.items():
            row[surface] = math.log10(vocabulary_rank(logits_row, token_id))
        rows.append(row)
    return pd.DataFrame(rows).set_index("layer")

j_rank_frame = rank_frame(j_logits)
logit_rank_frame = rank_frame(logit_logits)

fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(display_layers) * 0.45)))
sns.heatmap(j_rank_frame, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[0])
sns.heatmap(logit_rank_frame, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[1])
axes[0].set_title("J-lens: log10(vocabulary rank)")
axes[1].set_title("Logit lens: log10(vocabulary rank)")
for axis in axes:
    axis.set_xlabel("tracked token")
    axis.set_ylabel("layer")
plt.tight_layout()
plt.show()

### 7. Exact two-token coordinate swap

논문의 두-vector swap을 그대로 구현합니다. 각 layer에서 `V=[v_source, v_target]`, `c=V^†h`를 계산하고 두 coordinate를 교환합니다.

이 실험은 다음-token 분포만 비교합니다. 작은 모델에서 원하는 의미 변화가 생긴다는 보장은 없습니다. source token이 clean J-lens에서 강하게 활성화되어 있을수록 성공 가능성이 높습니다.

In [ ]:
SOURCE_SURFACE = " spider"
TARGET_SURFACE = " ant"

source_id = resolve_single_token(SOURCE_SURFACE)
target_id = resolve_single_token(TARGET_SURFACE)
assert source_id is not None and target_id is not None

workspace_start = int(len(lens.source_layers) * 0.40)
workspace_end = max(workspace_start + 1, int(len(lens.source_layers) * 0.75))
SWAP_LAYERS = lens.source_layers[workspace_start:workspace_end]
print("swap layers:", SWAP_LAYERS)

def token_unembedding_direction(token_id):
    weight = lens_model._lm_head.weight
    return weight[token_id].detach().float()

def j_direction(layer, token_id):
    w = token_unembedding_direction(token_id).cpu()
    return w @ lens.jacobians[layer].float()

swap_operators = {}
for layer in SWAP_LAYERS:
    source_direction = j_direction(layer, source_id)
    target_direction = j_direction(layer, target_id)
    V = torch.stack([source_direction, target_direction], dim=1).float()  # [d_model, 2]
    V_pinv = torch.linalg.pinv(V)  # [2, d_model]
    swap_operators[layer] = (V.to(DEVICE), V_pinv.to(DEVICE))

def make_coordinate_swap_hook(V, V_pinv):
    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        original_dtype = hidden.dtype
        h = hidden.float()
        coordinates = h @ V_pinv.T
        swapped_coordinates = coordinates.flip(dims=(-1,))
        patched = h + (swapped_coordinates - coordinates) @ V.T
        patched = patched.to(original_dtype)
        if isinstance(output, tuple):
            return (patched, *output[1:])
        return patched
    return hook

@contextmanager
def coordinate_swap_hooks():
    handles = []
    try:
        for layer, (V, V_pinv) in swap_operators.items():
            handles.append(
                lens_model.layers[layer].register_forward_hook(
                    make_coordinate_swap_hook(V, V_pinv)
                )
            )
        yield
    finally:
        for handle in handles:
            handle.remove()

prompt_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(DEVICE)
with torch.inference_mode():
    clean_next_logits = hf_model(input_ids=prompt_ids).logits[0, -1].float().cpu()
    with coordinate_swap_hooks():
        swapped_next_logits = hf_model(input_ids=prompt_ids).logits[0, -1].float().cpu()

def next_token_table(logits_row, label, k=10):
    probabilities = logits_row.softmax(dim=-1)
    values, indices = probabilities.topk(k)
    return pd.DataFrame(
        {
            "condition": label,
            "token": [tokenizer.decode([int(i)]) for i in indices],
            "probability": values.numpy(),
        }
    )

display(
    pd.concat(
        [
            next_token_table(clean_next_logits, "clean"),
            next_token_table(swapped_next_logits, f"{SOURCE_SURFACE} → {TARGET_SURFACE}"),
        ],
        ignore_index=True,
    )
)

delta = swapped_next_logits - clean_next_logits
delta_values, delta_ids = delta.topk(10)
display(
    pd.DataFrame(
        {
            "most increased token": [tokenizer.decode([int(i)]) for i in delta_ids],
            "logit change": delta_values.numpy(),
        }
    )
)

## Checks

아래 체크는 notebook 결과를 해석하기 전에 확인해야 합니다.

In [ ]:
checks = {
    "GPU runtime": DEVICE.type == "cuda",
    "model/lens d_model match": lens.d_model == lens_model.d_model,
    "J-lens layers returned": set(j_logits) == set(display_layers),
    "logit-lens layers returned": set(logit_logits) == set(display_layers),
    "finite clean logits": bool(torch.isfinite(clean_next_logits).all()),
    "finite swapped logits": bool(torch.isfinite(swapped_next_logits).all()),
    "swap changed distribution": not torch.allclose(clean_next_logits, swapped_next_logits),
}
display(pd.Series(checks, name="passed").to_frame())

if not checks["GPU runtime"]:
    print("GPU가 아니어도 결과는 계산되지만 Colab GPU를 권장합니다.")

## Optional: fit a tiny lens from scratch

이 단계는 **방법 확인용**입니다. 논문 품질의 lens는 128-token corpus 약 1,000개를 사용합니다. 아래 셀은 계산량을 줄이기 위해 Pythia 70M, 일부 layer, 소수 prompt만 사용하므로 결과 품질을 논문과 비교하면 안 됩니다.

실행하려면 `RUN_TINY_FIT = True`로 변경하세요. GPU 메모리가 부족하면 먼저 런타임을 다시 시작하고 Setup 셀과 이 셀만 실행하는 편이 안전합니다.

In [ ]:
RUN_TINY_FIT = False

if RUN_TINY_FIT:
    assert torch.cuda.is_available(), "tiny fitting도 Colab GPU를 권장합니다."
    tiny_id = "EleutherAI/pythia-70m-deduped"
    tiny_tokenizer = transformers.AutoTokenizer.from_pretrained(tiny_id)
    tiny_hf = transformers.AutoModelForCausalLM.from_pretrained(
        tiny_id, torch_dtype=torch.float16
    ).to("cuda")
    tiny_model = jlens.from_hf(tiny_hf, tiny_tokenizer, compile=False)

    tiny_prompts = [
        "Scientific instruments allow researchers to measure subtle changes in complex physical systems over long periods of time.",
        "During the expedition, the team recorded weather patterns, animal behavior, and changes in the surrounding landscape.",
        "A careful explanation should distinguish direct observations from assumptions and from conclusions inferred using prior knowledge.",
        "The history of a city can be studied through architecture, public records, oral traditions, maps, and archaeological evidence.",
    ]
    tiny_source_layers = list(range(0, tiny_model.n_layers - 1, 2))
    fitted_tiny_lens = jlens.fit(
        tiny_model,
        prompts=tiny_prompts,
        source_layers=tiny_source_layers,
        dim_batch=8,
        max_seq_len=64,
        skip_first=8,
    )
    fitted_tiny_lens.save("tiny_pythia_jacobian_lens.pt")
    print(fitted_tiny_lens)
    print("saved: tiny_pythia_jacobian_lens.pt")
else:
    print("Skipped. Set RUN_TINY_FIT=True to run the optional fitting demo.")

## Next Steps

- Prompt를 `currency of the country shaped like a boot`, rhyme planning, 간단한 산술 등으로 바꿔 비교합니다.
- `display_layers` 대신 특정 workspace band를 촘촘하게 선택합니다.
- clean J-lens에서 source token의 rank가 충분히 높은 경우에만 coordinate swap 결과를 해석합니다.
- Neuronpedia UI와 동일한 대화형 화면이 필요해진 뒤에 inference server/webapp을 연결합니다.
- 논문의 sparse non-negative J-space decomposition과 occupancy 분석은 이 notebook에 포함되지 않았습니다.

### References

- Paper: https://transformer-circuits.pub/2026/workspace/index.html
- Reference code: https://github.com/anthropics/jacobian-lens
- Neuronpedia integration: https://github.com/hijohnnylin/neuronpedia
- Pre-fitted lenses: https://huggingface.co/neuronpedia/jacobian-lens